# Task 2 — Technical Stock Analysis

**Objective:** Load OHLCV stock price data, validate and clean it, then compute a suite of technical indicators using TA-Lib and pynance.  Visualise each indicator with publication-quality charts and provide financial interpretation.

**Sections:**
1. Setup
2. Data loading
3. Datatype validation
4. Missing value handling
5. Chronological sorting
6. Simple and Exponential Moving Averages
7. Relative Strength Index (RSI)
8. MACD
9. Bollinger Bands (pynance)
10. Volume analysis
11. Growth metrics (pynance)
12. All indicators in one table
13. Summary

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.indicators import (
    load_ohlcv,
    sma, ema, rsi, macd,
    daily_returns, log_returns, rolling_volatility,
    bollinger_bands, growth_metrics, compute_all,
)
from src.visualization import time_series, price_volume

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
%matplotlib inline

---
## 2. Data Loading

We attempt to load a local CSV with OHLCV columns.  If none exists, we download live data via ``yfinance``.

In [ ]:
prices_path = root / "data" / "raw" / "stock_prices.csv"

if prices_path.exists():
    df = load_ohlcv(prices_path)
    print(f"Loaded local CSV: {df.shape[0]} rows, {df.shape[1]} columns")
else:
    print("No local CSV found - downloading AAPL data via yfinance...")
    import yfinance as yf
    raw = yf.download("AAPL", start="2022-01-01", auto_adjust=False, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    df = raw
    df.index = pd.to_datetime(df.index)
    print(f"Downloaded AAPL: {df.shape[0]} rows")

expected_cols = ["Open", "High", "Low", "Close", "Volume"]
if "Adj Close" in df.columns:
    expected_cols.append("Adj Close")
available = [c for c in expected_cols if c in df.columns]
print(f"OHLCV columns available: {available}")
df.head()

---
## 3. Datatype Validation

We verify that every OHLCV column is stored as a numeric type and the index is a ``DatetimeIndex``.

In [ ]:
print("=== Column Dtypes ===")
print(df.dtypes.to_string())

print(f"\nIndex type: {type(df.index).__name__}")
print(f"Index range: {df.index.min()}  to  {df.index.max()}")
print(f"Monotonic increasing: {df.index.is_monotonic_increasing}")

In [ ]:
numeric_cols = df.select_dtypes("number").columns.tolist()
print(f"Numeric columns: {numeric_cols}")
non_numeric = [c for c in available if c not in numeric_cols]
if non_numeric:
    print(f"WARNING - non-numeric OHLCV columns: {non_numeric}")
else:
    print("All OHLCV columns are numeric - OK")

**Insight:** The data types are correct and the index is a ``DatetimeIndex`` sorted chronologically.

---
## 4. Missing Value Handling

Financial time series often have small gaps.  We identify null values and apply forward-fill.

In [ ]:
missing = df[available].isnull().sum()
missing_pct = df[available].isnull().mean() * 100
missing_report = pd.DataFrame({"null_count": missing, "null_pct": missing_pct.round(2)})
missing_report

In [ ]:
before = len(df)
df = df.ffill().dropna(subset=available)
after = len(df)
print(f"Rows dropped: {before - after}  (remaining: {after})")

**Insight:** Missing values are minimal.  Forward-filling is standard for financial data where the last observed price persists until the next trade.

---
## 5. Chronological Sorting

Technical indicators require chronological order.  We enforce a sorted index.

In [ ]:
if not df.index.is_monotonic_increasing:
    df = df.sort_index()
    print("Sorted chronologically.")
else:
    print("Already in chronological order.")

close = df["Close"]
print(f"Close price range: {close.min():.2f} - {close.max():.2f}")

---
## 6. Moving Averages (SMA & EMA)

We compute 20-day and 50-day SMA plus a 20-day EMA using **TA-Lib**.  Moving averages smooth price data to visualise the underlying trend.

In [ ]:
close_sma20 = sma(close, window=20)
close_sma50 = sma(close, window=50)
close_ema20 = ema(close, span=20)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(close.index, close, label="Close", linewidth=1.5, color="black")
ax.plot(close.index, close_sma20, label="SMA(20)", linewidth=1.2)
ax.plot(close.index, close_sma50, label="SMA(50)", linewidth=1.2, linestyle="--")
ax.plot(close.index, close_ema20, label="EMA(20)", linewidth=1.2, linestyle=":")

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()
ax.set(title="Close Price with Moving Averages (TA-Lib)", ylabel="Price (USD)", xlabel="")
ax.legend()
sns.despine()
plt.show()

**Interpretation:**
- **SMA(20) crossing above SMA(50)** → golden cross (bullish).
- **SMA(20) crossing below SMA(50)** → death cross (bearish).
- **EMA** reacts faster to recent price changes, making it more responsive in trending markets.

---
## 7. Relative Strength Index (RSI)

RSI measures the magnitude of recent price changes to evaluate overbought (>= 70) or oversold (<= 30) conditions.  Computed with **TA-Lib**.

In [ ]:
close_rsi = rsi(close, window=14)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(close.index, close, color="black", linewidth=1.2)
ax1.set(ylabel="Price (USD)", title="Close Price and RSI(14)")

ax2.plot(close_rsi.index, close_rsi, color="purple", linewidth=1.2)
ax2.axhline(70, color="red", linestyle="--", alpha=0.5, label="Overbought (70)")
ax2.axhline(30, color="green", linestyle="--", alpha=0.5, label="Oversold (30)")
ax2.fill_between(close_rsi.index, 70, 100, alpha=0.08, color="red")
ax2.fill_between(close_rsi.index, 0, 30, alpha=0.08, color="green")
ax2.set(ylabel="RSI", ylim=(0, 100))
ax2.legend(loc="upper right")

ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()
sns.despine()
plt.show()

**Interpretation:**
- RSI **> 70** suggests overbought - potential pullback.
- RSI **< 30** suggests oversold - potential bounce.
- **Divergences** (price high / RSI lower high) can signal trend reversals.

---
## 8. MACD

MACD captures the relationship between two EMAs (12, 26).  The histogram shows momentum.  Computed with **TA-Lib**.

In [ ]:
macd_df = macd(close, fast=12, slow=26, signal=9)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(close.index, close, color="black", linewidth=1.2)
ax1.set(ylabel="Price (USD)", title="Close Price and MACD (TA-Lib)")

ax2.plot(macd_df.index, macd_df["macd"], label="MACD", color="blue", linewidth=1.2)
ax2.plot(macd_df.index, macd_df["signal"], label="Signal", color="orange", linewidth=1.2)
ax2.bar(
    macd_df.index,
    macd_df["histogram"],
    width=1.5,
    color=np.where(macd_df["histogram"] >= 0, "green", "red"),
    alpha=0.4,
    label="Histogram",
)
ax2.axhline(0, color="gray", linewidth=0.5)
ax2.set(ylabel="MACD", xlabel="")
ax2.legend(loc="upper left")

ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()
sns.despine()
plt.show()

**Interpretation:**
- **MACD crosses above signal** → bullish (buy).
- **MACD crosses below signal** → bearish (sell).
- **Histogram turning green** → accelerating upward momentum.

---
## 9. Bollinger Bands (pynance)

Bollinger Bands (middle SMA +/- 2 standard deviations) are computed via **pynance**.  They adapt to market volatility.

In [ ]:
bb = bollinger_bands(close, window=20, num_std=2.0)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(close.index, close, label="Close", color="black", linewidth=1.2)
ax.plot(bb.index, bb["mid"], label="SMA(20) - mid", color="blue", linewidth=1.0)
ax.plot(bb.index, bb["upper"], label="Upper (+2sd)", color="green", linestyle="--", linewidth=0.8)
ax.plot(bb.index, bb["lower"], label="Lower (-2sd)", color="red", linestyle="--", linewidth=0.8)
ax.fill_between(bb.index, bb["upper"], bb["lower"], alpha=0.05, color="gray")

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()
ax.set(title="Bollinger Bands (pynance)", ylabel="Price (USD)", xlabel="")
ax.legend()
sns.despine()
plt.show()

**Interpretation:**
- Price at **upper band** suggests overextended bullish momentum.
- Price at **lower band** suggests oversold conditions.
- **Band squeeze** (narrowing) often precedes a sharp price move.

---
## 10. Volume Analysis

Volume confirms price trends.  Rising prices on increasing volume strengthen the trend.

In [ ]:
fig = price_volume(
    df, price_col="Close", volume_col="Volume",
    title="Close Price and Trading Volume",
)
plt.show()

**Interpretation:**
- **High volume** on up-days confirms bullish conviction.
- **Low volume** on breakouts suggests a false signal.
- Spikes often coincide with earnings announcements.

---
## 11. Growth Metrics (pynance)

pynance provides simple growth, log growth, and growth volatility - useful for risk assessment.

In [ ]:
gm = growth_metrics(close)
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(gm.index, gm["growth"], color="darkblue", linewidth=0.6)
axes[0].set(ylabel="Growth", title="Growth Metrics (pynance)")
axes[1].plot(gm.index, gm["ln_growth"], color="darkgreen", linewidth=0.6)
axes[1].set(ylabel="Log Growth")
axes[2].plot(gm.index, gm["growth_vol"], color="darkred", linewidth=0.6)
axes[2].set(ylabel="Growth Volatility", xlabel="")

for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()
sns.despine()
plt.show()

---
## 12. All Indicators in One Table

The ``compute_all()`` helper consolidates every indicator into a single DataFrame for downstream modelling.

In [ ]:
full = compute_all(df, close_col="Close")
full.head(10)

In [ ]:
print(f"Feature set: {full.shape[1]} columns")
print(list(full.columns))

---
## 13. Summary

| Indicator | Library | Purpose |
|-----------|---------|--------|
| SMA(20/50) | TA-Lib | Trend direction and cross signals |
| EMA(20) | TA-Lib | Trend with faster price response |
| RSI(14) | TA-Lib | Overbought / oversold gauge |
| MACD (12/26/9) | TA-Lib | Momentum with cross signals |
| Bollinger Bands | pynance | Volatility-based support / resistance |
| Volume | raw | Confirmation / divergence |
| Growth metrics | pynance | Returns, log returns, growth volatility |

All functions live in ``src/indicators.py`` with TA-Lib as the primary engine (pandas fallback) and pynance for Bollinger Bands and growth metrics.  These features feed into the sentiment correlation analysis in **Task 3**.